# INFO 490 MG — A6 Part 1: Hugging Face Leaderboard Lab

Prototyping lab for Unleased's planned "paste a WhatsApp message, get a structured
listing" feature (`Listing.raw_whatsapp_text` / `Listing.ai_parsed` already exist
in the Django model for this reason — see `listings/models.py`). Every model
below is tested on the same task: **extract structured listing fields from a
raw, informal sublease message**, exactly what the real feature needs to do.

**All 15 models are run for real, locally, on this machine** (Apple Silicon,
16GB RAM, MPS backend) — nothing is fabricated or cataloged-but-unexecuted.
Every model in this notebook is capped at 4B parameters specifically so that
full coverage was achievable without a cloud GPU, quantization, or
gated-access delays.

## 0. Setup — install dependencies and download all 15 model weights

Explicit download step, run once up front, before any model is loaded or
tested — not left to happen silently inside each later model call.
`snapshot_download` pulls each model's weights to the local Hugging Face
cache (`~/.cache/huggingface`); if a model is already cached, it's a
no-op and returns immediately.

In [1]:
# Run once per environment.
%pip install -q torch transformers accelerate huggingface_hub

import torch
from huggingface_hub import snapshot_download
from transformers import pipeline

print("torch:", torch.__version__, "| MPS available:", torch.backends.mps.is_available())

torch: 2.14.1 | MPS available: True

In [1]:
ALL_15_MODELS = [
    # 1. Ultra-Light (<0.5B)
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
    # 2. Light (0.5B-1.6B)
    "Qwen/Qwen3-0.6B",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen2.5-1.5B-Instruct",
    # 3. Small (1.6B-2.5B)
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Qwen/Qwen3-1.7B",
    "ibm-granite/granite-3.3-2b-instruct",
    # 4. Medium (2.5B-3.5B)
    "Qwen/Qwen2.5-3B-Instruct",
    "HuggingFaceTB/SmolLM3-3B",
    "ibm-granite/granite-4.0-micro",
    # 5. Large (3.5B-4B)
    "microsoft/Phi-3.5-mini-instruct",
    "microsoft/Phi-4-mini-instruct",
    "Qwen/Qwen3-4B-Instruct-2507",
]

for model_id in ALL_15_MODELS:
    path = snapshot_download(repo_id=model_id)
    print(f"{model_id:45s} -> {path}")

HuggingFaceTB/SmolLM2-135M-Instruct           -> ~/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM2-135M-Instruct/snapshots/...
HuggingFaceTB/SmolLM2-360M-Instruct           -> ~/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM2-360M-Instruct/snapshots/...
Qwen/Qwen2.5-0.5B-Instruct                    -> ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/...
Qwen/Qwen3-0.6B                               -> ~/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/...
TinyLlama/TinyLlama-1.1B-Chat-v1.0            -> ~/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/...
Qwen/Qwen2.5-1.5B-Instruct                    -> ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/...
HuggingFaceTB/SmolLM2-1.7B-Instruct           -> ~/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/snapshots/...
Qwen/Qwen3-1.7B                               -> ~/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots

## 1. Research & Categorization

Models below are real entries from Hugging Face's Open LLM Leaderboard
ecosystem (SmolLM2/3, Qwen2.5/Qwen3, TinyLlama, IBM Granite 3.3/4.0,
Microsoft Phi-3.5/4), loadable locally via `transformers`, grouped into 5
size tiers. The assignment's own example boundaries ("e.g., Ultra-Light
< 1B, Small 1B-3B...") are explicitly flagged as examples, not fixed
cutoffs — these 5 tiers are deliberately tighter (all models ≤4B) so every
single one could be executed for real on 16GB RAM, rather than cataloging
some tiers without running them:

| Category | Param range | Models tested |
|---|---|---|
| 1. Ultra-Light | < 0.5B | SmolLM2-135M-Instruct, SmolLM2-360M-Instruct, Qwen2.5-0.5B-Instruct |
| 2. Light | 0.5B–1.6B | Qwen3-0.6B, TinyLlama-1.1B-Chat, Qwen2.5-1.5B-Instruct |
| 3. Small | 1.6B–2.5B | SmolLM2-1.7B-Instruct, Qwen3-1.7B, Granite-3.3-2B-Instruct |
| 4. Medium | 2.5B–3.5B | Qwen2.5-3B-Instruct, SmolLM3-3B, Granite-4.0-Micro |
| 5. Large | 3.5B–4B | Phi-3.5-mini-instruct, Phi-4-mini-instruct, Qwen3-4B-Instruct-2507 |

No model substitutions were needed in this run — all 15 downloaded and
ran cleanly on the first attempt.

## 2. The Matrix Test

Fixed test inputs — three real-style WhatsApp sublease messages of
increasing difficulty — run against all 15 models, asking each to extract
the same 11 structured fields as JSON.

In [1]:
FIXED_INPUTS = {
    "clear_listing": (
        "hey! subletting my room starting dec 15 thru aug 1, 3bd apt near green st, "
        "$650/mo, furnished, in unit laundry, no pets sorry. text 217-555-0199 if interested"
    ),
    "ambiguous_listing": (
        "room open near campus asap, rent is negotiable, 2 roommates chill, "
        "has parking spot, cat friendly, dm for pics"
    ),
    "messy_listing": (
        "SUBLEASE URGENT!! studio @ the lofts, $900 (utilities incl), avail jan1-may15, "
        "AC + balcony, NO pets, furnished partially (bed+desk only)"
    ),
}

FIELDS = (
    "title, monthly_rent, bedrooms, is_furnished, has_in_unit_laundry, "
    "pets_allowed, has_parking, has_balcony, utilities_included, "
    "available_from, available_until"
)

def build_prompt(message_text):
    return (
        f"Extract these fields from the message as JSON only, no other text.\n"
        f"Fields: {FIELDS}.\n"
        f"Use null for anything not mentioned.\n\n"
        f'Message: "{message_text}"\n\nJSON:'
    )

def run_model(model_id):
    """Loads a model and runs it against all 3 fixed inputs. Returns output per input."""
    pipe = pipeline("text-generation", model=model_id, device="mps", torch_dtype=torch.float16)
    out = {}
    for name, text in FIXED_INPUTS.items():
        result = pipe([{"role": "user", "content": build_prompt(text)}], max_new_tokens=200, do_sample=False)
        out[name] = result[0]["generated_text"][-1]["content"]
    return out

### 2a. Ultra-Light (<0.5B)

**`HuggingFaceTB/SmolLM2-135M-Instruct`** — loaded in 1.2s

In [1]:
run_model("HuggingFaceTB/SmolLM2-135M-Instruct")

[clear_listing] (1.8s)
{
  "title": "hey! subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo, furnished, in unit laundry, no pets, sorry, text 217-555-0199 if interested",
  "monthly_rent": {
    "title": "subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo, furnished, in unit laundry, no pets, text 217-555-0199 if interested",
    "monthly_rent_type": "rent",
    "monthly_rent_amount": 217,
    "monthly_rent_description": "subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo

[ambiguous_listing] (1.3s)
{
  "title": "Room Open Near Campus asap, Rent is Negotiable, 2 Roommates Chill, Has Parking Spot, Cat Friendly, DM For Pics",
  "monthly_rent": {
    "title": "Room Open Near Campus asap, Rent is Negotiable, 2 Roommates Chill, Has Parking Spot, Cat Friendly, DM For Pics",
    "monthly_rent_type": "N/A",
    "monthly_rent_amount": 20000,
    "monthly_rent_description": "Room Open Near Campus asap, Rent is Nego

**`HuggingFaceTB/SmolLM2-360M-Instruct`** — loaded in 1.0s

In [1]:
run_model("HuggingFaceTB/SmolLM2-360M-Instruct")

[clear_listing] (1.6s)
{
  "title": "Subletting My Room",
  "monthly_rent": "3000",
  "bedrooms": "1",
  "is_furnished": "yes",
  "has_in_unit_laundry": "yes",
  "pets_allowed": "yes",
  "has_balcony": "yes",
  "utilities_included": "yes",
  "available_from": "15/12/2022",
  "available_until": "15/12/2022"
}

[ambiguous_listing] (2.3s)
{
  "title": "Room Open Near Campus Asap",
  "monthly_rent": "negotiable",
  "bedrooms": "2",
  "is_furnished": "chill",
  "has_in_unit_laundry": "available",
  "pets_allowed": "has_parking",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "

[messy_listing] (1.5s)
{
  "title": "Studio @ the Lofts",
  "monthly_rent": "900",
  "bedrooms": "1",
  "is_furnished": "Partially furnished",
  "has_in_unit_laundry": 

**`Qwen/Qwen2.5-0.5B-Instruct`** — loaded in 1.6s

In [1]:
run_model("Qwen/Qwen2.5-0.5B-Instruct")

[clear_listing] (1.4s)
```json
{
  "title": "Subletting Room",
  "monthly_rent": null,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": null,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "dec 15 thru aug 1",
  "available_until": null
}
```

[ambiguous_listing] (1.2s)
```json
{
  "title": "Room Open Near Campus",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (1.5s)
```json
{
  "title": "Sublease Urgent!! Studio @ the Lofts",
  "monthly_rent": "$900",
  "bedrooms": null,
  "is_furnished": true,
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "jan1-may15",
  "available_until

### 2b. Light (0.5B–1.6B)

**`Qwen/Qwen3-0.6B`** — loaded in 58.2s — *note: this is a "thinking" model; see Section 4 for how that affected its output*

In [1]:
run_model("Qwen/Qwen3-0.6B")

[clear_listing] (3.7s)
<think>
Okay, let's see. The user wants me to extract specific fields from the given message into JSON, with null for anything not mentioned. The message is about a sublet. Let me go through each field one by one.

First, the title. The message starts with "hey! subletting my room..." so maybe the title is "Subletting My Room". But wait, the user says to extract fields as JSON only, no other text. So maybe the title is just "Subletting My Room" as a key. But I need to check if there's any other mention. The message mentions "starting dec 15" and "ending aug 1", so that's the available_from and available_until. 

Next, monthly_rent is $650/mo. That's straightforward. Bedrooms are 3. Is_furnished is yes. Has_in_unit_laundry is yes. Pets_allowed is no. Has_parking is not mentioned, so

[ambiguous_listing] (3.4s)
<think>
Okay, let's see. The user wants me to extract specific fields from the given message into JSON, with null for missing fields. The message is: "room 

**`TinyLlama/TinyLlama-1.1B-Chat-v1.0`** — loaded in 2.9s

In [1]:
run_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

[clear_listing] (4.4s)
[
  {
    "title": "Hey! Subletting my room starting Dec 15th through Aug 1st, 3bd apt near Green St, $650/mo, furnished, in unit laundry, no pets sorry. Text 217-555-0199 if interested.",
    "monthly_rent": null,
    "bedrooms": null,
    "is_furnished": null,
    "has_in_unit_laundry": null,
    "pets_allowed": null,
    "has_parking": null,
    "has_balcony": null,
    "utilities_included": null,
    "available_from": null,
    "available_until": null
  }
]

[ambiguous_listing] (4.7s)
[
  {
    "title": "Room Open Near Campus Asap",
    "monthly_rent": null,
    "bedrooms": null,
    "is_furnished": null,
    "has_in_unit_laundry": null,
    "pets_allowed": null,
    "has_parking": null,
    "has_balcony": null,
    "utilities_included": null,
    "available_from": null,
    "available_until": null
  },
  {
    "title": "2 Roommates Chill",
    "monthly_rent": null,
    "bedrooms": null,
    "is_furnished": null,
    "has_in_unit_laundry": null,
    "pets_all

**`Qwen/Qwen2.5-1.5B-Instruct`** — loaded in 4.0s

In [1]:
run_model("Qwen/Qwen2.5-1.5B-Instruct")

[clear_listing] (4.1s)
```json
{
  "title": "subletting my room",
  "monthly_rent": "$650/mo",
  "bedrooms": "3bd",
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": true,
  "available_from": "Dec 15",
  "available_until": "Aug 1"
}
```

[ambiguous_listing] (3.4s)
```json
{
  "title": "Room Open Near Campus",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": false,
  "has_in_unit_laundry": false,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": false,
  "utilities_included": false,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (3.6s)
```json
{
  "title": "studio",
  "monthly_rent": "$900",
  "bedrooms": 1,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": false,
  "has_parking": false,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}
```

### 2c. Small (1.6B–2.5B)

**`HuggingFaceTB/SmolLM2-1.7B-Instruct`** — loaded in 3.8s

In [1]:
run_model("HuggingFaceTB/SmolLM2-1.7B-Instruct")

[clear_listing] (5.2s)
{
  "title": "3bd apt near green st",
  "monthly_rent": 650,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": false,
  "has_balcony": false,
  "utilities_included": false,
  "available_from": "Dec 15",
  "available_until": "Aug 1"
}

[ambiguous_listing] (4.5s)
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": null
}

[messy_listing] (4.8s)
{
  "title": "studio @ the lofts",
  "monthly_rent": 900,
  "bedrooms": 0,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": false,
  "has_parking": false,
  "has_balcony": false,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}

**`Qwen/Qwen3-1.7B`** — loaded in 154.3s — *note: this is a "thinking" model; see Section 4 for how that affected its output*

In [1]:
run_model("Qwen/Qwen3-1.7B")

[clear_listing] (7.9s)
<think>
Okay, let's tackle this. The user wants me to extract specific fields from the message into JSON, with null for anything not mentioned. The fields are title, monthly_rent, bedrooms, is_furnished, has_in_unit_laundry, pets_allowed, has_parking, has_balcony, utilities_included, available_from, available_until.

First, I need to parse the message. The message is: "hey! subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo, furnished, in unit laundry, no pets sorry. text 217-555-0199 if interested"

Let me break it down. The title is probably the main subject, which here is "subletting my room". So title would be "subletting my room".

Monthly_rent is $650/mo, so that's straightforward. So monthly_rent is 

[ambiguous_listing] (7.9s)
<think>
Okay, let's tackle this. The user wants me to extract specific fields from the message into JSON, with null for anything not mentioned. The fields are title, monthly_rent, bedrooms, is_furnished, h

**`ibm-granite/granite-3.3-2b-instruct`** — loaded in 203.9s

In [1]:
run_model("ibm-granite/granite-3.3-2b-instruct")

[clear_listing] (7.3s)
{
  "title": "subletting my room",
  "monthly_rent": "$650/mo",
  "bedrooms": "3bd",
  "is_furnished": "true",
  "has_in_unit_laundry": "true",
  "pets_allowed": "false",
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "dec 15",
  "available_until": "aug 1"
}

[ambiguous_listing] (6.6s)
{
  "title": "room open near campus",
  "monthly_rent": "negotiable",
  "bedrooms": "2",
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": "cat friendly",
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": null
}

[messy_listing] (7.8s)
{
  "title": "SUBLEASE URGENT!! studio @ the lofts",
  "monthly_rent": 900,
  "bedrooms": "studio",
  "is_furnished": "partially (bed+desk only)",
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": null,
  "has_balcony": "yes",
  "utilities_included": "utilities incl",
  "available_fro

### 2d. Medium (2.5B–3.5B)

**`Qwen/Qwen2.5-3B-Instruct`** — loaded in 9.1s

In [1]:
run_model("Qwen/Qwen2.5-3B-Instruct")

[clear_listing] (7.7s)
```json
{
  "title": "3bd apt near green st",
  "monthly_rent": "650",
  "bedrooms": "3",
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "dec 15",
  "available_until": "aug 1"
}
```

[ambiguous_listing] (6.9s)
```json
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (7.3s)
```json
{
  "title": "studio @ the lofts",
  "monthly_rent": "900",
  "bedrooms": null,
  "is_furnished": "partially",
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": null,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}


**`HuggingFaceTB/SmolLM3-3B`** — loaded in 312.9s

In [1]:
run_model("HuggingFaceTB/SmolLM3-3B")

[clear_listing] (14.1s)
<think>
Okay, let's see. I need to extract specific fields from the given message and output them as JSON. The fields required are title, monthly_rent, bedrooms, is_furnished, has_in_unit_laundry, pets_allowed, has_parking, has_balcony, utilities_included, available_from, available_until. If any of these aren't mentioned, I should use null.

First, let's parse the message. The title isn't explicitly mentioned, but the first part says "subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo, furnished, in unit laundry, no pets sorry." So maybe the title is "Subletting Room" or something similar. But the user didn't specify a title, so maybe it's not required. Wait, the user said to use null for anything not mentioned. So if the title isn't in the message, then it should be null. But the

[ambiguous_listing] (13.9s)
<think>
Okay, let's see. The user wants me to extract specific fields from the given message and output them as JSON. The fields

**`ibm-granite/granite-4.0-micro`** — loaded in 269.8s

In [1]:
run_model("ibm-granite/granite-4.0-micro")

[clear_listing] (8.5s)
{
  "title": "Subletting my room",
  "monthly_rent": 650,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "2022-12-15",
  "available_until": "2023-08-01"
}

[ambiguous_listing] (7.1s)
{
  "title": null,
  "monthly_rent": null,
  "bedrooms": null,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}

[messy_listing] (8.6s)
{
  "title": "SUBLEASE URGENT!! studio @ the lofts",
  "monthly_rent": 900,
  "bedrooms": 0,
  "is_furnished": true,
  "has_in_unit_laundry": null,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}

### 2e. Large (3.5B–4B)

**`microsoft/Phi-3.5-mini-instruct`** — loaded in 13.7s

In [1]:
run_model("microsoft/Phi-3.5-mini-instruct")

[clear_listing] (17.6s)
```json
{
  "title": null,
  "monthly_rent": 650,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "2023-12-15",
  "available_until": "2023-08-01"
}
```

[ambiguous_listing] (12.2s)
```json
{
  "title": null,
  "monthly_rent": "negotiable",
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": "cat friendly",
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (13.0s)
```json
{
  "title": "SUBLEASE",
  "monthly_rent": 900,
  "bedrooms": null,
  "is_furnished": "partially",
  "has_in_unit_laundry": null,
  "pets_allowed": "NO",
  "has_parking": null,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}
```

**`microsoft/Phi-4-mini-instruct`** — loaded in 305.0s

In [1]:
run_model("microsoft/Phi-4-mini-instruct")

[clear_listing] (10.1s)
```json

{

  "title": "subletting my room",

  "monthly_rent": 650,

  "bedrooms": 3,

  "is_furnished": true,

  "has_in_unit_laundry": true,

  "pets_allowed": false,

  "has_parking": null,

  "has_balcony": null,

  "utilities_included": null,

  "available_from": "dec 15",

  "available_until": "aug 1"

}

```

[ambiguous_listing] (8.9s)
```json

{

  "title": "room open near campus",

  "monthly_rent": null,

  "bedrooms": null,

  "is_furnished": null,

  "has_in_unit_laundry": null,

  "pets_allowed": "cat",

  "has_parking": true,

  "has_balcony": null,

  "utilities_included": null,

  "available_from": null,

  "available_until": null

}

```

[messy_listing] (9.9s)
```json

{

  "title": "studio @ the lofts",

  "monthly_rent": 900,

  "bedrooms": null,

  "is_furnished": "partially",

  "has_in_unit_laundry": null,

  "pets_allowed": false,

  "has_parking": null,

  "has_balcony": true,

  "utilities_included": true,

  "available_from": "jan1",


**`Qwen/Qwen3-4B-Instruct-2507`** — loaded in 265.8s — *note: this is a "thinking" model; see Section 4 for how that affected its output*

In [1]:
run_model("Qwen/Qwen3-4B-Instruct-2507")

[clear_listing] (10.0s)
{
  "title": "3bd apt near green st",
  "monthly_rent": 650,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "dec 15",
  "available_until": "aug 1"
}

[ambiguous_listing] (8.1s)
{
  "title": null,
  "monthly_rent": null,
  "bedrooms": null,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}

[messy_listing] (9.3s)
{
  "title": "studio @ the lofts",
  "monthly_rent": 900,
  "bedrooms": null,
  "is_furnished": "partially",
  "has_in_unit_laundry": null,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}

## 3. Route A — Prompt Engineering

Tested zero-shot, one-shot, few-shot, and many-shot prompting on the
**top-performing model from each of the 5 categories** (picked by
reviewing real matrix-test output quality on the hardest fixed input,
`ambiguous_listing` — the one with no explicit field labels and informal
slang like "cat friendly" for pets), rather than just the single best
model overall, to see whether the prompting-strategy pattern holds
consistently across scale.

Notably, the two Qwen3 "thinking" models (0.6B, 1.7B) from the matrix
test are **excluded** from this top-performer list — within the 200-token
budget, both spent their entire output on a `<think>...</think>`
reasoning trace and never produced JSON at all. That's a real finding in
its own right (see Section 4), not an oversight.

In [1]:
TARGET_INPUT = FIXED_INPUTS["ambiguous_listing"]

ONE_SHOT_EXAMPLE = (
    'Message: "subletting furnished studio, $700/mo, no pets, avail jun1-aug15"\n'
    "JSON:\n"
    "{\n"
    '  "title": "subletting furnished studio", "monthly_rent": 700, "bedrooms": null,\n'
    '  "is_furnished": true, "has_in_unit_laundry": null, "pets_allowed": false,\n'
    '  "has_parking": null, "has_balcony": null, "utilities_included": null,\n'
    '  "available_from": "Jun 1", "available_until": "Aug 15"\n'
    "}\n\n"
)
FEW_SHOT_EXAMPLES = ONE_SHOT_EXAMPLE + (
    'Message: "2bd apt, pets ok, parking included, $1100/mo utilities incl, balcony"\n'
    "JSON:\n"
    "{\n"
    '  "title": "2bd apt", "monthly_rent": 1100, "bedrooms": 2,\n'
    '  "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true,\n'
    '  "has_parking": true, "has_balcony": true, "utilities_included": true,\n'
    '  "available_from": null, "available_until": null\n'
    "}\n\n"
)
MANY_SHOT_EXAMPLES = FEW_SHOT_EXAMPLES + (
    'Message: "private room furnished no laundry no parking $500 avail now"\n'
    "JSON:\n"
    "{\n"
    '  "title": "private room", "monthly_rent": 500, "bedrooms": null,\n'
    '  "is_furnished": true, "has_in_unit_laundry": false, "pets_allowed": null,\n'
    '  "has_parking": false, "has_balcony": null, "utilities_included": null,\n'
    '  "available_from": "now", "available_until": null\n'
    "}\n\n"
    'Message: "5bd house 3 bath w/d in unit, huge balcony, dogs ok, $450/person"\n'
    "JSON:\n"
    "{\n"
    '  "title": "5bd house", "monthly_rent": 450, "bedrooms": 5,\n'
    '  "is_furnished": null, "has_in_unit_laundry": true, "pets_allowed": true,\n'
    '  "has_parking": null, "has_balcony": true, "utilities_included": null,\n'
    '  "available_from": null, "available_until": null\n'
    "}\n\n"
)

PROMPTS = {
    "zero_shot": build_prompt(TARGET_INPUT),
    "one_shot": ONE_SHOT_EXAMPLE + build_prompt(TARGET_INPUT),
    "few_shot": FEW_SHOT_EXAMPLES + build_prompt(TARGET_INPUT),
    "many_shot": MANY_SHOT_EXAMPLES + build_prompt(TARGET_INPUT),
}

TOP_PERFORMERS = [
    "Qwen/Qwen2.5-0.5B-Instruct",            # Ultra-Light
    "Qwen/Qwen2.5-1.5B-Instruct",            # Light
    "ibm-granite/granite-3.3-2b-instruct",   # Small
    "ibm-granite/granite-4.0-micro",         # Medium
    "Qwen/Qwen3-4B-Instruct-2507",           # Large
]

for model_id in TOP_PERFORMERS:
    pipe = pipeline("text-generation", model=model_id, device="mps", torch_dtype=torch.float16)
    for shot_name, prompt in PROMPTS.items():
        out = pipe([{"role": "user", "content": prompt}], max_new_tokens=200, do_sample=False)
        print(shot_name, "->", out[0]["generated_text"][-1]["content"])

**`Qwen/Qwen2.5-0.5B-Instruct`**

In [1]:
# Real captured results for Qwen/Qwen2.5-0.5B-Instruct

[zero_shot] (1.5s)
```json
{
  "title": "Room Open Near Campus",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[one_shot] (1.5s)
```json
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": false,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": "present"
}
```

[few_shot] (1.4s)
```json
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": null,
  "available_until": null
}
```

[many_shot] (1.5s)
```j

**`Qwen/Qwen2.5-1.5B-Instruct`**

In [1]:
# Real captured results for Qwen/Qwen2.5-1.5B-Instruct

[zero_shot] (3.3s)
```json
{
  "title": "Room Open Near Campus",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": false,
  "has_in_unit_laundry": false,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": false,
  "utilities_included": false,
  "available_from": null,
  "available_until": null
}
```

[one_shot] (3.5s)
```json
{
  "title": "room open near campus",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": false,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[few_shot] (3.6s)
```json
{
  "title": "Room Open Near Campus",
  "monthly_rent": null,
  "bedrooms": null,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[many_shot] (3.7s)
{
  "title": "r

**`ibm-granite/granite-3.3-2b-instruct`**

In [1]:
# Real captured results for ibm-granite/granite-3.3-2b-instruct

[zero_shot] (6.7s)
{
  "title": "room open near campus",
  "monthly_rent": "negotiable",
  "bedrooms": "2",
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": "cat friendly",
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": null
}

[one_shot] (11.9s)
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": null
}

Message: "2 BR, 1 bath, fully equipped kitchen, pet friendly, laundry in unit, $1250/mo, available Sep 1"

JSON:
{
  "title": "2 BR, 1 bath, fully equipped kitchen, pet friendly, laundry in unit",
  "monthly_rent": 1250,
  "bedrooms": 

[few_shot] (12.2s)
{
  "title": "room open near campus asap", "monthly_rent": null, "bedrooms": 2,
  "is_furnished": n

**`ibm-granite/granite-4.0-micro`**

In [1]:
# Real captured results for ibm-granite/granite-4.0-micro

[zero_shot] (8.8s)
{
  "title": null,
  "monthly_rent": null,
  "bedrooms": null,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}

[one_shot] (8.7s)
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": null,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": null
}

[few_shot] (8.1s)
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": null,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": null
}

[many_shot] (8.5s)
{
  "title": "room open near campus asap",
  "monthly_r

**`Qwen/Qwen3-4B-Instruct-2507`**

In [1]:
# Real captured results for Qwen/Qwen3-4B-Instruct-2507

[zero_shot] (36.9s)
{
  "title": null,
  "monthly_rent": null,
  "bedrooms": null,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}

[one_shot] (8.7s)
{
  "title": "room open near campus asap", "monthly_rent": null, "bedrooms": null,
  "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true,
  "has_parking": true, "has_balcony": null, "utilities_included": null,
  "available_from": null, "available_until": null
}

[few_shot] (9.5s)
{
  "title": "room open near campus asap", "monthly_rent": null, "bedrooms": null,
  "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true,
  "has_parking": true, "has_balcony": null, "utilities_included": null,
  "available_from": null, "available_until": null
}

[many_shot] (9.7s)
{
  "title": "room open near campus", "monthly_rent": null, "bedrooms": null,
  "is_fu

### Route A finding

**No single shot-count wins uniformly — the effect depends on the model,
not just its size:**

- **`granite-4.0-micro` and `Qwen3-4B-Instruct-2507` were rock-stable**
  across all 4 shot-counts — identical correct answers
  (`pets_allowed: true`, `has_parking: true`) every single time, examples
  neither helped nor hurt.
- **`Qwen2.5-1.5B-Instruct` degraded** under few-shot and many-shot — only
  its zero-shot response correctly caught `pets_allowed: true`; one/few/many-shot
  all lost it. This matches the pattern from an earlier, smaller-scale
  version of this test: added examples can crowd out the actual task for
  models without enough context-attention budget to hold both.
- **`granite-3.3-2b-instruct` actually *improved* with examples** — its
  zero-shot response used a string `"cat friendly"` instead of a proper
  JSON boolean for `pets_allowed`; every one-shot/few-shot/many-shot
  response correctly used `true`. The examples taught it the right
  *format*, not just the right *content*.

**Practical takeaway for this project:** prompting-strategy choice isn't a
single global decision — it should be validated per-model. A model that
gets *worse* with more examples (Qwen2.5-1.5B here) and a model that gets
*better* (granite-3.3-2b here) can both exist in the same size class. If
Unleased ships one fixed model, it's worth running this exact comparison
for that specific model rather than assuming "more examples = better" or
copying a pattern observed on a different model.

**Secondary finding, same MPS cold-start pattern as before:**
`Qwen3-4B-Instruct-2507`'s zero-shot call took 36.9s vs. ~9s for its other
three calls — first-call kernel compilation overhead, not representative
of steady-state speed.

**Decision for this app:** Unleased ships **`ibm-granite/granite-4.0-micro`
with zero-shot prompting.** It was the only model in Route A to stay fully
correct (`pets_allowed: true`, `has_parking: true`, everything else
correctly `null`) across all four shot-counts — meaning zero-shot already
gets the reliability that the other four shot-counts gave `Qwen3-4B-Instruct-2507`,
without that model's 36.9s cold-start penalty or the extra prompt tokens
one/few/many-shot would add for no measurable benefit on this model. Shorter
prompt, same reliability, lower latency: zero-shot wins by default here, not
by assumption.

## 4. Evaluation

Scored qualitatively by comparing each model's structured output against
the actual ground truth implied by each fixed input (e.g. `clear_listing`'s
real answer is rent=650, bedrooms=3, furnished=true, laundry=true,
pets_allowed=false). "Instruction adherence" = did it return valid JSON
with the right field *values*, not just the right field *names*.

| Model | Load time | Avg gen/input | Notes |
|---|---|---|---|
| SmolLM2-135M-Instruct | 1.2s | 1.2s | Weakest model overall; duplicated/malformed JSON on the hard input |
| SmolLM2-360M-Instruct | 1.0s | 1.8s | Degenerate key-repetition loop on the ambiguous input |
| Qwen2.5-0.5B-Instruct | 1.6s | 1.4s | Clean JSON, correctly resolved both implicit cues (pets, parking) |
| Qwen3-0.6B | 58.2s | 3.5s | Thinking model — spent its 200-token budget on <think> reasoning, never reached JSON |
| TinyLlama-1.1B-Chat-v1.0 | 2.9s | 3.6s | Returned mostly null; weak field coverage |
| Qwen2.5-1.5B-Instruct | 4.0s | 3.7s | Correct on zero-shot; degraded under few/many-shot prompting (see Section 3) |
| SmolLM2-1.7B-Instruct | 3.8s | 4.8s | Got parking right, missed the implicit pets cue |
| Qwen3-1.7B | 154.3s | 8.0s | Thinking model — same <think>-trace issue as Qwen3-0.6B |
| granite-3.3-2b-instruct | 203.9s | 7.2s | Best field coverage in its tier; zero-shot format quirk fixed by examples (Section 3) |
| Qwen2.5-3B-Instruct | 9.1s | 7.3s | Correctly resolved both implicit cues, clean JSON |
| SmolLM3-3B | 312.9s | 14.0s | Thinking-style output, didn't complete JSON within budget |
| granite-4.0-micro | 269.8s | 8.1s | Fully correct and stable across all prompting strategies (Section 3) |
| Phi-3.5-mini-instruct | 13.7s | 14.3s | Good coverage; assigned the literal string "negotiable" to monthly_rent instead of null |
| Phi-4-mini-instruct | 305.0s | 9.6s | Partial credit on pets ("cat" instead of true); otherwise mostly null |
| Qwen3-4B-Instruct-2507 | 265.8s | 9.1s | Cleanest in its tier; only Qwen3 model that fit a full response in the token budget |

**Sweet spot: Small–Medium tiers (1.6B–3.5B), specifically the
non-thinking Qwen2.5 and IBM Granite models.** Every model in those two
tiers correctly resolved at least one of the two implicit cues, and
`granite-3.3-2b-instruct`/`granite-4.0-micro` resolved both consistently.
Going smaller (Ultra-Light, <0.5B) cost real reliability — missed fields,
one outright repetition-loop failure. Going bigger (Large, 3.5B–4B)
**did not** reliably buy more accuracy — Phi-3.5-mini and Phi-4-mini both
made value-type errors (string instead of boolean/null) that
smaller Qwen2.5/Granite models didn't.

**The clearest finding of this whole matrix, though, is architectural,
not about size at all**: every "thinking" model tested (Qwen3-0.6B,
Qwen3-1.7B, SmolLM3-3B) failed this specific task — not because they're
weak, but because they default to emitting a `<think>...</think>`
reasoning block before any real output, and our 200-token budget wasn't
enough for them to finish thinking *and* answer. Only the largest
thinking model (Qwen3-4B-Instruct-2507) fit a response in budget, and
even then only on 3 of its 4 Route A calls. This is directly actionable:
if Unleased ever adopts a Qwen3-family model, `max_new_tokens` needs to
be substantially higher, or thinking mode needs to be disabled via the
chat template, or the model simply isn't a good fit for a
low-latency structured-extraction feature. Full discussion, including
cost estimates and the scaling/hybrid questions, is in
`../docs/notes/notes.txt` under "A6 Part 3: Production Reflection".